<a href="https://colab.research.google.com/github/lilhawkeye2002-ux/notebooks/blob/main/Whisper_Bulk_Transcriber_AltTimestampMethod.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎙️ Whisper Bulk Transcriber — Timestamped Transcript Variant

Transcribe audio and video files into **millisecond-accurate, timestamped** plain text using OpenAI's Whisper `large-v2` model.

This variant differs from the standard transcriber in one key way: the `.txt` output includes **DTW-aligned timestamps** on every line — sourced directly from when words are spoken in the audio, not rounded token predictions.

---

## What makes this different

| | Standard `Whisper_Bulk_Transcriber` | This notebook |
|---|---|---|
| `.txt` content | Plain text, no timestamps | `[HH:MM:SS.mmm --> HH:MM:SS.mmm]  spoken text` |
| Timestamp source | Model token predictions (20 ms steps) | Dynamic Time Warping on cross-attention weights |
| How accurate | ±20–60 ms | Word-boundary aligned to actual audio |
| `.srt` / `.vtt` | Included | Included (also DTW-refined) |

### How the timestamps work

Enabling `word_timestamps=True` triggers Whisper's `timing.py` module to run **Dynamic Time Warping (DTW)** on the decoder's cross-attention weights — observing which audio frames the model attended to when producing each word. It then overwrites each segment's `start`/`end` with word-boundary-aligned float values. This is measurement-based alignment, not prediction.

---

## How To Use (3 Steps)

### Step 1 — Enable a free GPU
**Runtime → Change runtime type → Hardware accelerator → T4 GPU**

### Step 2 — Run the cell for the first time
Click **▶** on the code cell below. It will create the upload folder and stop with:
> *"Directory created. Upload your audio/video files there, then run this cell again."*

### Step 3 — Upload files and run again
1. Open the **Files panel** (📁 in the left sidebar)
2. Navigate to `/content/bulk_process_audios_here`
3. **Drag and drop** your audio or video files in
4. Click **▶** again — transcription starts automatically

---

## Output

For each source file, three outputs are saved to `/content/audio_transcription/`:

| Format | Content |
|---|---|
| `.txt` | `[00:00:00.000 --> 00:00:04.820]  Welcome everyone, today we're going to...` |
| `.srt` | Subtitle format with DTW-refined timestamps |
| `.vtt` | Web caption format with DTW-refined timestamps |

All files are zipped to `/content/all_transcribed_files.zip` — **download before closing the session**.

---

## Supported formats
| Audio | Video |
|-------|-------|
| `.mp3` `.wav` `.m4a` `.flac` | `.mp4` `.mov` `.avi` `.mkv` |
| `.aac` `.ogg` `.wma` `.opus` | `.webm` |
| `.aiff` `.amr` `.au` | |

> 🎬 Video files are handled automatically — the audio track is extracted before transcription.

---

## Troubleshooting

| Issue | Fix |
|---|---|
| Transcription very slow | Switch to GPU: **Runtime → Change runtime type → T4 GPU** |
| `ffmpeg` not found | Add a cell above: `!apt install -y ffmpeg` |
| GPU out of memory | **Runtime → Restart session**, then re-run |
| File skipped `[SKIP]` | File is empty (0 bytes) or unreadable — re-upload it |
| Session expired before download | Re-run the cell — all files are re-transcribed and re-zipped |

In [ ]:
# @title ## Whisper Bulk Transcriber — Timestamped
# @markdown ### Outputs `.txt` with DTW-aligned millisecond timestamps, plus `.srt` and `.vtt`.
# @markdown Final zip saved to `/content/all_transcribed_files.zip`.

# ── Phase 0: Constants & Minimal Imports ─────────────────────────────────────
# Only stdlib modules needed here — no pip installs required for directory check.

import os
import sys

BULK_DIR   = "/content/bulk_process_audios_here"
OUTPUT_DIR = "/content/audio_transcription"
ZIP_PATH   = "/content/all_transcribed_files.zip"
USE_MODEL  = "large-v2"

VIDEO_EXTENSIONS = {'.mp4', '.avi', '.mov', '.mkv', '.webm'}

SUPPORTED_EXTENSIONS = {
    '.mp3', '.wav', '.m4a', '.flac', '.aac', '.ogg', '.wma',
    '.aiff', '.opus', '.amr', '.au',
} | VIDEO_EXTENSIONS

TRANSCRIBED_EXTS = {'.txt', '.srt', '.vtt'}

# ── Phase 1: Directory Check (runs FIRST — before any installs) ───────────────
# Instant and dependency-free. Saves the user from waiting through pip installs
# and model loading when there is nothing to do yet.

os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.exists(BULK_DIR):
    os.makedirs(BULK_DIR)
    sys.exit(
        f"\nDirectory created: '{BULK_DIR}'\n"
        f"Upload your audio/video files there, then run this cell again.\n"
    )

_all_in_dir = [
    os.path.join(BULK_DIR, f) for f in os.listdir(BULK_DIR)
    if os.path.isfile(os.path.join(BULK_DIR, f))
]
audio_files = sorted([
    f for f in _all_in_dir
    if os.path.splitext(f)[1].lower() in SUPPORTED_EXTENSIONS
])

if not audio_files:
    sys.exit(
        f"\nNo supported audio/video files found in '{BULK_DIR}'.\n"
        f"Supported formats: {', '.join(sorted(SUPPORTED_EXTENSIONS))}\n"
        f"Upload files there and run this cell again.\n"
    )

print(f"Found {len(audio_files)} file(s) to transcribe. Loading dependencies...")

# ── Phase 2: Full Imports ─────────────────────────────────────────────────────
# Only reached when files are confirmed present.

import re
import subprocess
import tempfile
import unicodedata
import zipfile

# ── Phase 3: Library Check / Install ─────────────────────────────────────────

def _install_libraries():
    cmd = [
        sys.executable, "-m", "pip", "install", "-q",
        "--root-user-action=ignore",
        "openai-whisper", "openai", "pydub", "ffmpeg-python"
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print("[ERROR] pip install failed:")
        print(result.stderr)
        sys.exit(1)

try:
    import whisper
    import torch
    import pydub   # noqa: F401
    import ffmpeg  # noqa: F401  (package: ffmpeg-python, imports as ffmpeg)
    print("Libraries already installed. Skipping installation.")
except ImportError:
    print("Installing required libraries...")
    _install_libraries()
    try:
        import whisper
        import torch
        import pydub   # noqa: F401
        import ffmpeg  # noqa: F401
        print("Libraries installed and imported successfully.")
    except ImportError as e:
        print(f"[ERROR] Import failed after installation: {e}")
        sys.exit(1)

# ── Phase 4: ffmpeg Binary Pre-flight Check ───────────────────────────────────

_ffmpeg_check = subprocess.run(
    ["ffmpeg", "-version"], capture_output=True
)
if _ffmpeg_check.returncode != 0:
    print("[ERROR] ffmpeg binary not found on PATH.")
    print("Fix: add a cell above with   !apt install -y ffmpeg   and run it first.")
    sys.exit(1)
print("ffmpeg binary confirmed.")

# ── Phase 5: Per-file Pre-validation ─────────────────────────────────────────

valid_audio_files = []
for _f in audio_files:
    _name = os.path.basename(_f)
    if not os.access(_f, os.R_OK):
        print(f"[SKIP] Not readable (check permissions): {_name}")
        continue
    if os.path.getsize(_f) == 0:
        print(f"[SKIP] Empty file (0 bytes): {_name}")
        continue
    valid_audio_files.append(_f)

if not valid_audio_files:
    sys.exit("[ERROR] No valid audio files remain after validation. Aborting.")

audio_files = valid_audio_files
print(f"{len(audio_files)} file(s) passed validation.")

# ── Phase 6: Model Load ───────────────────────────────────────────────────────

from whisper.utils import format_timestamp, get_writer  # noqa: E402

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if DEVICE == "cpu":
    _total_secs = 0.0
    _duration_known = True
    for _af in audio_files:
        _probe = subprocess.run(
            [
                "ffprobe", "-v", "error",
                "-show_entries", "format=duration",
                "-of", "default=noprint_wrappers=1:nokey=1",
                _af,
            ],
            capture_output=True, text=True,
        )
        try:
            _total_secs += float(_probe.stdout.strip())
        except ValueError:
            _duration_known = False
            break

    _EST_SLOWDOWN = 20
    _border = "!" * 64
    print(f"\n{_border}")
    print("  WARNING: NO GPU DETECTED — TRANSCRIPTION WILL BE")
    print("  OVERWHELMINGLY SLOW (roughly 1 transcribed line per minute).")
    print()
    if _duration_known and _total_secs > 0:
        _audio_mins = _total_secs / 60
        _est_mins   = (_total_secs * _EST_SLOWDOWN) / 60
        _est_hrs    = _est_mins / 60
        print(f"  Total audio duration : ~{_audio_mins:.0f} min")
        if _est_hrs >= 1:
            print(f"  Estimated CPU time   : ~{_est_hrs:.1f} hours  (could be longer)")
        else:
            print(f"  Estimated CPU time   : ~{_est_mins:.0f} minutes  (could be longer)")
        print()
    print("  STRONGLY RECOMMENDED: switch to a GPU runtime first.")
    print("  Runtime > Change runtime type > Hardware accelerator > T4 GPU")
    print(f"{_border}\n")

    _answer = input(
        "Continue on CPU anyway? This will take an extremely long time. [y/N]: "
    ).strip().lower()

    if _answer != "y":
        sys.exit(
            "\nAborted. To enable GPU:\n"
            "  Runtime > Change runtime type > Hardware accelerator > T4 GPU\n"
            "Then run this cell again.\n"
        )
    print("Acknowledged. Continuing on CPU — this will take a very long time.\n")

print(f"Loading '{USE_MODEL}' model on {DEVICE}...")
try:
    model = whisper.load_model(USE_MODEL, device=DEVICE)
    print(f"Model '{USE_MODEL}' loaded on {DEVICE}.")
except Exception as e:
    print(f"[ERROR] Failed to load Whisper model: {e}")
    sys.exit(1)

# ── Phase 7: LiveFileWriter (console-only) ────────────────────────────────────
#
# Streams Whisper's verbose output to the console in real-time.
# The .txt file is written AFTER transcription completes using the
# DTW-aligned segment data from result['segments'] — not by parsing printed text.
# _WHISPER_SEGMENT_RE is not needed here and is intentionally absent.

class LiveFileWriter:
    """Tees sys.stdout to the console only. .txt is written after transcription."""

    def __init__(self, stream):
        self.stream = stream

    def write(self, data):
        if data is None:
            return
        self.stream.write(data)
        self.stream.flush()

    def writelines(self, datas):
        for d in datas:
            self.write(d)

    def close(self):
        pass  # no file handle to close

    def __getattr__(self, name):
        return getattr(self.stream, name)


# ── Phase 7b: Video Audio Extraction Helper ───────────────────────────────────

def _extract_audio_from_video(video_path: str) -> str:
    """Extract audio track from a video file to a temp WAV; return its path."""
    tmp = tempfile.NamedTemporaryFile(
        suffix=".wav",
        prefix="_whisper_extract_",
        delete=False,
    )
    tmp_path = tmp.name
    tmp.close()

    cmd = [
        "ffmpeg", "-y",
        "-i", video_path,
        "-vn",
        "-acodec", "pcm_s16le",
        "-ar", "16000",
        "-ac", "1",
        tmp_path,
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        try:
            os.unlink(tmp_path)
        except OSError:
            pass
        err_detail = proc.stderr.strip().splitlines()[-1] if proc.stderr.strip() else "unknown ffmpeg error"
        raise RuntimeError(f"Audio extraction failed: {err_detail}")

    return tmp_path


# ── Phase 8: Timestamped .txt Writer ─────────────────────────────────────────
#
# Writes the .txt file from result['segments'] AFTER transcription completes.
#
# Why this is more accurate than parsing verbose console text:
#   word_timestamps=True causes Whisper's timing.py to run Dynamic Time Warping
#   on the decoder's cross-attention weights, then overwrite:
#       segment["start"] = words[0]["start"]
#       segment["end"]   = words[-1]["end"]
#   These float values are word-boundary-aligned to the actual audio — not the
#   decoder's timestamp token predictions (which have fixed 20 ms resolution).
#
# Why the .txt is NOT cleared before transcription:
#   If transcription fails, the previous successful .txt is preserved.
#   _write_timestamped_txt opens with "w" mode, truncating on successful write.

def _write_timestamped_txt(segments: list, output_path: str) -> None:
    """Write a timestamped plain-text transcript from DTW-aligned Whisper segments."""
    with open(output_path, "w", encoding="utf-8") as f:
        for seg in segments:
            start = format_timestamp(seg["start"], always_include_hours=True)
            end   = format_timestamp(seg["end"],   always_include_hours=True)
            text  = seg["text"].strip()
            if text:
                f.write(f"[{start} --> {end}]  {text}\n")


# ── Phase 9: Transcription Options ───────────────────────────────────────────

_options = {
    "task": "transcribe",
    "fp16": DEVICE == "cuda",
    "best_of": 5,
    "beam_size": 5,
    "temperature": (0.0, 0.2, 0.4, 0.6, 0.8, 1.0),
    "condition_on_previous_text": True,
    "initial_prompt": None,
    "language": None,
    "no_speech_threshold": 0.4,
    "logprob_threshold": -1.5,
    "compression_ratio_threshold": 2.2,
    # Enables DTW cross-attention alignment (timing.py).
    # Overwrites seg['start']/seg['end'] with word-boundary-aligned timestamps,
    # replacing the decoder's 20 ms-resolution token predictions with values
    # derived from which audio frames the model actually attended to per word.
    # Memory overhead is per-segment (hooks removed after each segment).
    "word_timestamps": True,
}

# ── Phase 10: Transcription Loop ─────────────────────────────────────────────

results      = {}
failed_files = {}
original_stdout = sys.stdout
n = len(audio_files)

for i, audio_path in enumerate(audio_files, 1):
    raw_stem    = os.path.splitext(os.path.basename(audio_path))[0]
    output_stem = unicodedata.normalize("NFC", raw_stem)
    txt_path    = os.path.join(OUTPUT_DIR, output_stem + ".txt")

    print(f"\n[{i}/{n}] Processing: {os.path.basename(audio_path)}")

    # --- Video → audio extraction --------------------------------------------
    _temp_audio = None
    transcribe_path = audio_path

    if os.path.splitext(audio_path)[1].lower() in VIDEO_EXTENSIONS:
        print(f"  Video file detected — extracting audio track via ffmpeg...")
        try:
            _temp_audio     = _extract_audio_from_video(audio_path)
            transcribe_path = _temp_audio
            print(f"  Audio extracted successfully.")
        except RuntimeError as e:
            msg = str(e)
            print(f"[FAIL] {os.path.basename(audio_path)}: {msg}")
            failed_files[audio_path] = msg
            continue

    # .txt is NOT pre-cleared — if transcription fails, the previous successful
    # .txt is preserved. _write_timestamped_txt truncates on successful write.

    live_writer = LiveFileWriter(original_stdout)
    result = None

    try:
        sys.stdout = live_writer
        result = whisper.transcribe(model, transcribe_path, verbose=True, **_options)

    except RuntimeError as e:
        msg = str(e)
        if "out of memory" in msg.lower():
            torch.cuda.empty_cache()
            msg = (
                "GPU out of memory. "
                "Try a shorter audio file, or restart the runtime to free VRAM: "
                "Runtime > Restart session."
            )
        else:
            msg = f"RuntimeError: {e}"
        failed_files[audio_path] = msg
        print(f"\n[FAIL] {os.path.basename(audio_path)}: {msg}")

    except Exception as e:
        msg = str(e)
        failed_files[audio_path] = msg
        print(f"\n[FAIL] {os.path.basename(audio_path)}: {msg}")

    finally:
        sys.stdout = original_stdout
        live_writer.close()
        if _temp_audio and os.path.exists(_temp_audio):
            os.unlink(_temp_audio)

    if result is None:
        continue

    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    if not result.get("segments"):
        print(f"  No speech detected in: {os.path.basename(audio_path)}")

    results[audio_path] = result

    # Write .txt from DTW-aligned segment data.
    if result.get("segments"):
        _write_timestamped_txt(result["segments"], txt_path)
        print(f"  Saved: {output_stem}.txt")

    # Write .srt and .vtt. These also benefit from DTW-refined boundaries
    # because get_writer reads result['segments'] which word_timestamps=True
    # has already overwritten with word-aligned start/end values.
    norm_audio_path = os.path.join(
        os.path.dirname(audio_path),
        output_stem + os.path.splitext(audio_path)[1]
    )

    for fmt in ("srt", "vtt"):
        try:
            fix_vtt = (
                fmt == "vtt"
                and bool(result.get("segments"))
                and result["segments"][0].get("start") == 0
            )
            if fix_vtt:
                result["segments"][0]["start"] += 1 / 1000

            writer = get_writer(fmt, OUTPUT_DIR)
            writer(result, norm_audio_path)

            if fix_vtt:
                result["segments"][0]["start"] = 0

            print(f"  Saved: {output_stem}.{fmt}")

        except Exception as e:
            print(f"  [WARN] Could not write .{fmt} for '{output_stem}': {e}")

# ── Phase 11: Zip All Transcribed Files ──────────────────────────────────────

if not results:
    print("\n[ERROR] No files transcribed successfully. Nothing to zip.")
else:
    _files_to_zip = sorted([
        os.path.join(OUTPUT_DIR, fname)
        for fname in os.listdir(OUTPUT_DIR)
        if os.path.splitext(fname)[1].lower() in TRANSCRIBED_EXTS
    ])

    if not _files_to_zip:
        print("\n[WARN] Output directory has no .txt/.srt/.vtt files to zip.")
    else:
        with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED, allowZip64=True) as zf:
            for fpath in _files_to_zip:
                zf.write(fpath, arcname=os.path.basename(fpath))

        _zip_mb = os.path.getsize(ZIP_PATH) / (1024 * 1024)
        print(
            f"\nZipped {len(_files_to_zip)} file(s) -> {ZIP_PATH} "
            f"({_zip_mb:.2f} MB)"
        )
        print(
            "REMINDER: Download your zip before the Colab session ends. "
            "/content/ is erased when the runtime disconnects."
        )

# ── Phase 12: Final Summary ───────────────────────────────────────────────────

print("\n" + "=" * 52)
print("TRANSCRIPTION SUMMARY")
print(f"  Total files : {n}")
print(f"  Succeeded   : {len(results)}")
print(f"  Failed      : {len(failed_files)}")
if failed_files:
    print("\nFailed files:")
    for _path, _reason in failed_files.items():
        print(f"  x {os.path.basename(_path)}")
        print(f"    Reason: {_reason}")
print("=" * 52)
